### COT Report data scraper

https://github.com/NDelventhal/cot_reports

In [110]:
import pandas as pd
import cot_reports as cot
from matplotlib import pyplot as plt
import numpy as np
import re
#%matplotlib widget
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML


### <u> 1. Instructions for using cot_reports </u>


From https://github.com/NDelventhal/cot_reports

#### Example: cot_hist()
df = cot.cot_hist(cot_report_type= 'traders_in_financial_futures_futopt')
#### cot_hist() downloads the historical bulk file for the specified report type, in this example the Traders in Financial Futures Futures-and-Options Combined report. Returns the data as dataframe.

#### Example: cot_year()
df = cot.cot_year(year = 2020, cot_report_type = 'traders_in_financial_futures_fut')
#### cot_year() downloads the single year file of the specified report type and year. Returns the data as dataframe.

#### Example for collecting data of a few years, here from 2017 to 2020, of a specified report:
df = pd.DataFrame()
begin_year = 2017
end_year = 2020
for i in range(begin_year, end_year + 1):
    single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_futopt')) 
    df = df.append(single_year, ignore_index=True)

#### Example: cot_all()
df = cot.cot_all(cot_report_type='legacy_fut')
#### cot_all() downloads the historical bulk file and all remaining single year files of the specified report type.  Returns the data as dataframe.

### 2. Download and Compile COT Data into a pandas dataframe

In [111]:
def cot_reader (start, end):
    df_list = []
    begin_year = start
    end_year = end
    for i in range(begin_year, end_year + 1):
        single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_fut')) 
        df_list.append(single_year)
    
    df = pd.concat(df_list, ignore_index=True)
    
    df.rename(columns = {"Market and Exchange Names" : "Market", 
                     "As of Date in Form YYMMDD" : "Datetime", 
                     "Open Interest (All)" : "OI", 
                     "As of Date in Form YYYY-MM-DD" : "Date"}, inplace=True )
    
    df["Datetime"] = pd.to_datetime(df["Datetime"], format = '%y%m%d')
    
    df.sort_values("Datetime", ascending = False, inplace = True)
    
    return df

In [112]:
#try legacy_fut
#try all instead of year

In [113]:
df = cot_reader(2022, 2025)

Selected: legacy_fut
Downloaded single year data from: 2022
Selected: legacy_fut
Downloaded single year data from: 2023
Selected: legacy_fut
Downloaded single year data from: 2024
Selected: legacy_fut
Downloaded single year data from: 2025


### 3. Create a list of markets to trade 

##### This is based on personal preference, please ignore if you're looking at all markets or edit as you like

In [114]:
keywords = ["gold", "silver", "platinum", "palladium", "copper", "Lithium", "crude", "heating", "oil", "Nat Gas",
"rbob", "brent", "cocoa", "corn", "oat", "wheat", "soybean", "soy bean", "feed" , "Hogs", "live", "OJ", "coffee", "cotton", "sugar","E-Mini", "Micro","WTI-PHYSICAL",
    "Russell",
    "S&P 500",
    "NASDAQ",
    "Dow Jones",
    "Nikkei",
    "FTSE",
    "DAX",
    "CAC",
    "SMI",
    "Hang Seng",
    "Shanghai",
    "Treasury", "UST", "Bond", "EURO" , "Peso", "Brazilian", "Swiss", "Canadian", "British", "Japanese", "New Zealand", "Rand", 
    "bitcoin" , "ether","SOFR", "vix"
           ]

In [115]:
unique_markets = []

for keyword in keywords:
    filtered_df = df[df["Market"].str.contains(keyword, case=False)]
    
    unique_values = filtered_df["Market"].unique()
    
    unique_markets.extend(unique_values)
    

In [116]:
remove_items = [
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    'EURO SHORT TERM RATE - CHICAGO MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTY-PAID - COMMODITY EXCHANGE INC.',
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH EURODOLLARS - CHICAGO MERCANTILE EXCHANGE',
    'COFFEE CALENDAR SPREAD OPTIONS - ICE FUTURES U.S.',
    'WHEAT-HRW - CHICAGO BOARD OF TRADE',
    'WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE',
    'BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF TRADE',
    'CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE',
    'CORN CSO - CHICAGO BOARD OF TRADE',
    'MARINE .5% FOB USGC/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'WTI-BRENT SPREAD OPTION - NEW YORK MERCANTILE EXCHANGE',
    'WTI-BRENT CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'USGC HSFO-PLATTS/BRENT 1ST LN - ICE FUTURES ENERGY DIV',
    'TRANSCONTINENTAL GAS- STATION 85 (ZONE 4) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - VENTURA (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-TEXOK (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (BASIS) - ICE FUTURES ENERGY DIV',
    'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COLUMBIA GAS CO. - TCO POOL (APPALACHIA) (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-MID-CONTINENT POOL PIN (BASIS) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - DEMARCATION POOL (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (INDEX) - ICE FUTURES ENERGY DIV',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'ONEOK GAS TRANSPORTATION BASIS - ICE FUTURES ENERGY DIV',
    'NATURAL GAS INDEX: EP SAN JUAN - ICE FUTURES ENERGY DIV',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'NATURAL GAS HENRY LD1 FIXED - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PENULTIMATE ICE - ICE FUTURES ENERGY DIV',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
    'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-TMX WCS 1A INDEX - ICE FUTURES ENERGY DIV',
    'CRUDE DIFF-TMX SW 1A INDEX - ICE FUTURES ENERGY DIV',
    'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM - ICE FUTURES ENERGY DIV',
    'MT BELV NAT GASOLINE OPIS - NEW YORK MERCANTILE EXCHANGE',
     'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P FINANCIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P UTILITIES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P 400 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P ENERGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P CONSU STAPLES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P TECHNOLOGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P HEALTH CARE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE', 
     'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE', 
     'DOW JONES U.S. REAL ESTATE IDX - CHICAGO BOARD OF TRADE',
     'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',  
     'NIKKEI STOCK AVERAGE YEN DENOM - CHICAGO MERCANTILE EXCHANGE',
     'COLUMBIA GULF TRANSMISSION CO. -  MAINLINE POOL - ICE FUTURES ENERGY DIV',
     'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COPPER-GRADE #1 - COMMODITY EXCHANGE INC.',
    'LITHIUM HYDROXIDE - COMMODITY EXCHANGE INC.',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
    'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MARINE FUEL OIL 0.5% FOB USGC - ICE FUTURES ENERGY DIV',
    'GULF JET NY HEAT OIL SPR - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
 'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM BARGES - ICE FUTURES ENERGY DIV',
    'EUR STYLE NATURAL GAS OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GAS LD1 for GDD -TEXOK - ICE FUTURES ENERGY DIV',
 'NAT GAS ICE LD1 - ICE FUTURES ENERGY DIV',
 'NATURAL GAS CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
 'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
 'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
 'BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MICRO GOLD - COMMODITY EXCHANGE INC.',
    'WTI 1st Line-Brent 1st Line - ICE FUTURES ENERGY DIV',
    'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'NFX CS5TC CAPESIZE 5T/C AVG - NASDAQ FUTURES',
    'NFX PM4TC PANAMAX 4T/C AVG - NASDAQ FUTURES',
    'NORTHWEST PIPELINE - CANADIAN BORDER (BASIS) - ICE FUTURES ENERGY DIV',
    'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS CUSHING/WTI 1ST - ICE FUTURES ENERGY DIV',
    'DUTCH TTF NAT GAS CAL MONTH - NEW YORK MERCANTILE EXCHANGE',
    'TRANSCONTINENTAL GAS - ZONE 6 (NY) (BASIS) - ICE FUTURES ENERGY DIV',
    'GULF COAST UNL 87 GAS M2 PL RB - NEW YORK MERCANTILE EXCHANGE',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS INDEX: ALGONQUIN CITY GATES - ICE FUTURES ENERGY DIV',
    'HOT ROLLED COIL STEEL - NEW YORK MERCANTILE EXCHANGE',
    '3.5% FUEL OIL RDAM CRACK SPR - NEW YORK MERCANTILE EXCHANGE',
    'HENRY HUB PENULTIMATE NAT GAS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GASLNE OPIS MT B NONTET FP - ICE FUTURES ENERGY DIV',
     'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'SOUTH AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE',
    'Nano Bitcoin - LMX LABS LLC',
    'NANO ETHER - LMX LABS LLC',
    '2 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'WCS OIL NET ENERGY MONTHLY IND - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
    'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'WTI  HOUSTON ARGUS/WTI TR MO - NEW YORK MERCANTILE EXCHANGE',
    'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE',
    'HOUSTON SHIP CHANNEL (INDEX) - ICE FUTURES ENERGY DIV',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE',
     'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
     'NAT GAS ICE PEN - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ERCOT Houston 345KV Hub RT 7x8 - ICE FUTURES ENERGY DIV',
    'ERCOT Houston 345KV RT OFF FIX - ICE FUTURES ENERGY DIV',
    'ERCOT HOUSTON 345KV RT PK FIX - ICE FUTURES ENERGY DIV',
     'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
 'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
  'GULF # 6 FUEL OIL CRACK - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) (BASIS) - ICE FUTURES ENERGY DIV',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) - ICE FUTURES ENERGY DIV',
 "WAHA HUB - WEST TEXAS DELIVERED/BUYER'S INDEX - ICE FUTURES ENERGY DIV",
 '5 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'MICRO 10 YEAR YIELD - CHICAGO BOARD OF TRADE', 
 'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
 'MICRO SING FOB MARINE FUEL .5% - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
 '5 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',
 'ARGUS WTI HOUSTON/WTI TRADE MO - ICE FUTURES ENERGY DIV',
 'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
 '10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '5-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '2-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 'UST BOND - CHICAGO BOARD OF TRADE',
 'ULTRA UST 10Y - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE',
 'ULTRA UST BOND - CHICAGO BOARD OF TRADE'
 'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'FUEL OIL USGC HSFO PLATTS BALM - ICE FUTURES ENERGY DIV',
    'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'E-MINI S&P REAL ESTATE INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
    'EUROPEAN PROPANE CIF ARA - NEW YORK MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTYUNPAID - COMMODITY EXCHANGE INC.',
    'E-MINI S&P COMMUNICATION INDEX - CHICAGO MERCANTILE EXCHANGE',
    'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
    'RANDOM LENGTH LUMBER - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',





    
    
]


In [117]:
for item in remove_items:
    if item in unique_markets:
        unique_markets.remove(item)

In [118]:
sorted(list(dict.fromkeys(unique_markets))) #this is the market list for making graphs.

['AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE',
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'COCOA - ICE FUTURES U.S.',
 'COFFEE C - ICE FUTURES U.S.',
 'COPPER- #1 - COMMODITY EXCHANGE INC.',
 'CORN - CHICAGO BOARD OF TRADE',
 'COTTON NO. 2 - ICE FUTURES U.S.',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE',
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE',
 'GOLD - COMMODITY EXCHANGE INC.',
 'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE',
 'LEAN HOGS - CHICAGO MERCA

In [119]:
#remove duplicates
unique_markets = list(dict.fromkeys(unique_markets))

### 4. Create Open Interest Index Value for a Commodity 

In [120]:
#create new DF for this part of the analysis
df2 = df.sort_values(['Market', 'Datetime'], ascending = [True, True])

In [121]:
#Group markerss and add Open Interest Index Column

group = df2.groupby("Market")["OI"]
df2["OI_Index"] = group.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

## Forumla for indexing:
#zi = (xi – min(x)) / (max(x) – min(x)) * 100 = (12 – 12) / (68 – 12) * 100 = 0


### 5. See if a market has a unique value based on your keyword or search term


In [122]:
def number_of_markets (keyword_or_phrase):
    
    filtered = df[df["Market"].str.contains(keyword_or_phrase, case=False)]
    filtered = filtered["Market"].unique().tolist()
    
    return filtered

In [123]:
number_of_markets("cocoa")

['COCOA - ICE FUTURES U.S.']

### Retail OI indexing

In [124]:
df2.columns.to_list()

['Market',
 'Datetime',
 'Date',
 'CFTC Contract Market Code',
 'CFTC Market Code in Initials',
 'CFTC Region Code',
 'CFTC Commodity Code',
 'OI',
 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
 'Open Interest (Old)',
 'Noncommercial Positions-Long (Old)',
 'Noncommercial Positions-Short (Old)',
 'Noncommercial Positions-Spreading (Old)',
 'Commercial Positions-Long (Old)',
 'Commercial Positions-Short (Old)',
 'Total Reportable Positions-Long (Old)',
 'Total Reportable Positions-Short (Old)',
 'Nonreportable Positions-Long (Old)',
 'Nonreportable Positions-Short (Old)',
 'Open Interest (Other)',
 'Noncommercial Positions-Long (Other)',
 'Noncommercial Positions-Short (Other)'

In [125]:
cols = ['Market', 'Datetime' , 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
        'OI_Index', "OI"]

In [126]:
df3 = df2[cols].copy()

In [127]:
df3.iloc[0, 1:13]

Datetime                                   2022-01-04 00:00:00
Noncommercial Positions-Long (All)                       46144
Noncommercial Positions-Short (All)                      38652
Noncommercial Positions-Spreading (All)                  65066
Commercial Positions-Long (All)                         164971
Commercial Positions-Short (All)                        194790
 Total Reportable Positions-Long (All)                  276181
Total Reportable Positions-Short (All)                  298508
Nonreportable Positions-Long (All)                       44505
Nonreportable Positions-Short (All)                      22178
OI_Index                                                   0.0
OI                                                      320686
Name: 1891, dtype: object

In [128]:
row_test = df3.iloc[0, 1:10] #list of column headers and 1st row of data

In [129]:
df3["Net Retail Position"] = df3["Nonreportable Positions-Long (All)"] - df3["Nonreportable Positions-Short (All)"]

In [130]:
group2 = df3.groupby("Market")["Net Retail Position"]
df3["Retail_Index"] = group2.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [131]:
df3["Net Commercial Position"] = df3["Commercial Positions-Long (All)"] - df3["Commercial Positions-Short (All)"]


In [132]:
group3 = df3.groupby("Market")["Net Commercial Position"]
df3["Commercial_Index"] = group3.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [133]:
df3["Net Traders Position"] = df3["Noncommercial Positions-Long (All)"] - df3["Noncommercial Positions-Short (All)"]


In [134]:
group4 = df3.groupby("Market")["Net Traders Position"]
df3["Traders_Index"] = group4.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [135]:
df3.tail()

,Market,Datetime,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Noncommercial Positions-Spreading (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Total Reportable Positions-Long (All),Total Reportable Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All),OI_Index,OI,Net Retail Position,Retail_Index,Net Commercial Position,Commercial_Index,Net Traders Position,Traders_Index
53715,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,2025-07-15,308915,146488,832334,857803,1046199,1999052,2025021,70047,44078,86.893643,2069099,25969,53.697383,-188396,84.886469,162427,10.439257
53714,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,2025-07-22,308963,155632,777798,855128,1036984,1941889,1970414,71415,42890,79.694836,2013304,28525,58.889972,-181856,87.367642,153331,6.489198
53713,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,2025-07-29,318449,162426,803864,836644,1016236,1958957,1982526,69916,46347,81.703586,2028873,23569,48.821713,-179592,88.226568,156023,7.658235
53712,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,2025-08-05,310559,168730,820082,830714,1001272,1961355,1990084,75069,46340,82.677835,2036424,28729,59.304404,-170558,91.653926,141829,1.494300
55428,XRP - CHICAGO MERCANTILE EXCHANGE,2025-07-22,3615,3609,352,0,0,3967,3961,87,93,NaN,4054,-6,NaN,0,NaN,6,NaN


### Summarise Latest Week's Data in a Table

In [136]:
OI_condition = df3[(df3['OI_Index'] >= 80) | (df3['OI_Index']<=20)]
Retail_condition = df3[(df3['Retail_Index'] >= 80) | (df3['Retail_Index']<=20)]
Commercial_condition = df3[(df3['Commercial_Index'] >= 80) | (df3['Commercial_Index']<=20)]
Date_coundition =df3["Datetime"].max() 

In [137]:
# Get the most recent date
most_recent_date = df3['Datetime'].max()

# Filter the DataFrame for the most recent date and conditions

summary_table = df3.loc[(df3['Datetime'] == most_recent_date) & 
                        (((df3['OI_Index'] >= 80) | (df3['OI_Index'] <= 20)) |
                         ((df3['Retail_Index'] >= 80) | (df3['Retail_Index'] <= 20)) |
                         ((df3['Commercial_Index'] >= 80) | (df3['Commercial_Index'] <= 20))),
                        ['Market', 'OI_Index', 'Retail_Index', 'Commercial_Index']]


In [138]:
summary_table_filtered = summary_table[summary_table['Market'].isin(unique_markets)]
summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_90128/3186465317.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


In [139]:
# Function to apply color to cells based on conditions
def color_cells(val):
    if val >= 80:
        return 'background-color: #4D9F6B'
    elif val <= 20:
        return 'background-color: #C46A33'
    else:
        return ''
    
columns_format = ['OI_Index', 'Retail_Index', 'Commercial_Index']

# Apply the style to the DataFrame using the 'applymap' function
styled_summary = summary_table_filtered.style.applymap(color_cells, subset=columns_format)

# Display the styled DataFrame
styled_summary


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_90128/1291443741.py:13: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_summary = summary_table_filtered.style.applymap(color_cells, subset=columns_format)


,Market,OI_Index,Retail_Index,Commercial_Index
54699,BITCOIN,54.565330,53.184878,83.214604
53883,COCOA,2.577561,40.266463,80.354522
54007,COFFEE C,14.256082,24.914463,48.484127
54069,COPPER- #1,30.211771,90.845795,42.211320
46445,CORN,48.667089,60.105645,82.129708
49411,COTTON NO. 2,54.933174,7.182155,86.075028
54980,E-MINI S&P 500,2.251066,67.388550,44.915486
54346,EURO FX,90.330588,74.936720,25.486033
56057,EURO FX/BRITISH POUND XRATE,93.900198,88.405479,36.814369
50024,FEEDER CATTLE,99.981386,5.345601,0.000000


### Mapping symbols from yf to COT CFTC data

In [140]:
mapping = {'GOLD - COMMODITY EXCHANGE INC.' : 'GC=F',
 'SILVER - COMMODITY EXCHANGE INC.': 'SI=F',
 'PLATINUM - NEW YORK MERCANTILE EXCHANGE' : 'PL=F',
 'PALLADIUM - NEW YORK MERCANTILE EXCHANGE' : 'PA=F',
 'COPPER- #1 - COMMODITY EXCHANGE INC.' : "HG=F",
 'SOYBEAN OIL - CHICAGO BOARD OF TRADE' : "ZL=F",
 'SOYBEANS - CHICAGO BOARD OF TRADE': "ZS=F",
 'SOYBEAN MEAL - CHICAGO BOARD OF TRADE' : "ZM=F",
 'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE':'CL=F',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE' : 'RB=F',
 'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE' : 'NG=F',
 'CORN - CHICAGO BOARD OF TRADE' : "ZC=F",
 'OATS - CHICAGO BOARD OF TRADE' : "ZO=F",
 'WHEAT-SRW - CHICAGO BOARD OF TRADE' : 'ZW=F',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE' : "GF=F",
 'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE' : "HE=F",
 'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE' : "LE=F",
 'COCOA - ICE FUTURES U.S.' : "CC=F",
 'COFFEE C - ICE FUTURES U.S.' : "KC=F",
 'COTTON NO. 2 - ICE FUTURES U.S.' : "CT=F",
 'SUGAR NO. 11 - ICE FUTURES U.S.' : "SB=F",
 'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE' : 'RTY=F',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE' : 'ES=F',
 'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'ETH-USD',
 'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE' : 'BTC-USD',
 'MICRO GOLD - COMMODITY EXCHANGE INC.': 'MGC=F',
 'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE' : 'RTY=F',
 'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE' : 'MNQ=F',
 'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE' : 'NKD=F',
 'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE' : '6A=F',
 'UST 5Y NOTE - CHICAGO BOARD OF TRADE' : 'ZF=F',
 'UST 2Y NOTE - CHICAGO BOARD OF TRADE' : 'ZT=F',
 'UST 10Y NOTE - CHICAGO BOARD OF TRADE' : 'ZN=F',
 'UST BOND - CHICAGO BOARD OF TRADE' : 'ZB=F',
 'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': '6M=F',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE' : '6L=F',
 'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE' : '6S=F', 
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE' : '6C=F',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE' : '6E=F',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE' : '6B=F',
 'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE' : "6J=F",
 'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': '6N=F',
 'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE' : '6Z=F',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE' : "BTC=F",
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE' : 'ETH=F',
 'VIX FUTURES - CBOE FUTURES EXCHANGE' : '^VIX',
 }

In [141]:
#mapping.items maps keys and values in tuple pairs
#list()puts the tuples in a list
#pdDataframe puts that in a df, with said column names
mapping_df = pd.DataFrame(list(mapping.items()), columns=['Market', 'YF_Symbol'])

In [142]:
market_categories = {
    'GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'SILVER - COMMODITY EXCHANGE INC.': 'Metals',
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'COPPER- #1 - COMMODITY EXCHANGE INC.': 'Metals',
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': 'Softs',
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'CORN - CHICAGO BOARD OF TRADE': 'Grains',
    'OATS - CHICAGO BOARD OF TRADE': 'Grains',
    'WHEAT-SRW - CHICAGO BOARD OF TRADE': 'Grains',
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'COCOA - ICE FUTURES U.S.': 'Softs',
    'COFFEE C - ICE FUTURES U.S.': 'Softs',
    'COTTON NO. 2 - ICE FUTURES U.S.': 'Softs',
    'SUGAR NO. 11 - ICE FUTURES U.S.': 'Softs',
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST BOND - CHICAGO BOARD OF TRADE': 'Financials',
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
     'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE' : 'Currencies',
    'VIX FUTURES - CBOE FUTURES EXCHANGE': 'Indices',
}


In [143]:
#map the categories to the market in the mapping df
mapping_df["group"] = mapping_df["Market"].map(market_categories)

In [144]:
df4 = df3.copy()

In [145]:
df4 = pd.merge(df4, mapping_df, on="Market", how='left')

In [146]:
df4.rename(columns = {"Datetime" : "Date"}, inplace = True)

In [147]:
df4['group'].fillna('Financials', inplace=True)

## Create RSI Function


In [148]:
def rsi(data, periods=10):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

## Download Price Data

In [149]:
import yfinance as yf
from datetime import date, timedelta
start_date="2022-01-01"
end_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
tickers = mapping_df['YF_Symbol'].to_list()
commods = yf.download(tickers, start=start_date, end=end_date)




/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_90128/1709997627.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  commods = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  45 of 45 completed


In [150]:

# Create daily price dataset for all markets
daily_prices = []
for market_name in unique_markets:
    # Get the YF_Symbol for this market
    market_symbol = mapping_df[mapping_df['Market'] == market_name]['YF_Symbol'].iloc[0] if len(mapping_df[mapping_df['Market'] == market_name]) > 0 else None
    
    if market_symbol and market_symbol in commods['Close'].columns:
        # Get daily prices for this symbol
        market_daily = commods['Close'][market_symbol].reset_index()
        market_daily.columns = ['Date', 'Close']
        market_daily['Market'] = market_name
        market_daily['YF_Symbol'] = market_symbol
        daily_prices.append(market_daily)

# Combine all daily prices
daily_price_df = pd.concat(daily_prices, ignore_index=True)
daily_price_df = daily_price_df.dropna()

# Calculate 200-day moving average for daily data
daily_price_df['200MA'] = daily_price_df.groupby('Market')['Close'].rolling(window=200).mean().reset_index(0, drop=True)

In [151]:
# Calculate RSI for daily price data
daily_price_df['RSI'] = daily_price_df.groupby('Market')['Close'].transform(lambda x: rsi(x))

# Create a combined dataset that includes both weekly COT data and daily price data
# We'll add a 'data_type' column to distinguish between weekly COT data and daily price data

# Add data_type to existing df6 (weekly COT data)
df6_weekly = df6.copy()
df6_weekly['data_type'] = 'weekly_cot'

# Prepare daily price data to match df6 structure
daily_price_expanded = daily_price_df.copy()
daily_price_expanded['data_type'] = 'daily_price'

# Add missing columns from df6 to daily_price_expanded (fill with NaN for COT-specific data)
cot_columns = ['OI', 'OI_Index', 'Retail_Index', 'Commercial_Index', 'Traders_Index', 
               'Net Retail Position', 'Net Commercial Position', 'Net Traders Position',
               'Noncommercial Positions-Long (All)', 'Noncommercial Positions-Short (All)',
               'Commercial Positions-Long (All)', 'Commercial Positions-Short (All)',
               'Nonreportable Positions-Long (All)', 'Nonreportable Positions-Short (All)']

for col in cot_columns:
    if col not in daily_price_expanded.columns:
        daily_price_expanded[col] = None

# Add group column to daily price data by merging with mapping_df
if 'group' not in daily_price_expanded.columns:
    daily_price_expanded = pd.merge(daily_price_expanded, mapping_df[['Market', 'group']], on='Market', how='left')

# Add missing columns from daily data to df6_weekly if needed
if 'YF_Symbol' not in df6_weekly.columns:
    # Merge YF_Symbol from mapping_df
    df6_weekly = pd.merge(df6_weekly, mapping_df[['Market', 'YF_Symbol']], on='Market', how='left')

# Ensure both dataframes have the same column structure
common_columns = ['Market', 'Date', 'Close', '200MA', 'RSI', 'YF_Symbol', 'data_type', 'group'] + cot_columns

# Reorder columns for consistency
df6_weekly = df6_weekly.reindex(columns=common_columns, fill_value=None)
daily_price_expanded = daily_price_expanded.reindex(columns=common_columns, fill_value=None)

# Combine the datasets
df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)
df6_combined = df6_combined.sort_values(['Market', 'Date']).reset_index(drop=True)

print(f"Combined dataset created with {len(df6_combined)} total records")
print(f"Weekly COT data: {len(df6_weekly)} records")
print(f"Daily price data: {len(daily_price_expanded)} records")


Combined dataset created with 50109 total records
Weekly COT data: 8598 records
Daily price data: 41511 records


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_90128/4148037189.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)


In [152]:
daily_price_df.head()


,Date,Close,Market,YF_Symbol,200MA,RSI
2,2022-01-03,1799.400024,GOLD - COMMODITY EXCHANGE INC.,GC=F,NaN,NaN
3,2022-01-04,1814.000000,GOLD - COMMODITY EXCHANGE INC.,GC=F,NaN,NaN
4,2022-01-05,1824.599976,GOLD - COMMODITY EXCHANGE INC.,GC=F,NaN,NaN
5,2022-01-06,1788.699951,GOLD - COMMODITY EXCHANGE INC.,GC=F,NaN,NaN
6,2022-01-07,1797.000000,GOLD - COMMODITY EXCHANGE INC.,GC=F,NaN,NaN


#### Add Price data to COT dataframe

In [153]:
adj_close = commods['Close']


In [154]:
adj_close.reset_index(inplace=True)

In [155]:
#takes the prices and organises them by date by symbol in two columns, rather than date on the side and symbol across.
#id_vars makes date the identifier variable, and this column stays as it is in the new dataframe

adj_close = pd.melt(commods['Close'].reset_index(), id_vars='Date', var_name='YF_Symbol', value_name='Close')


In [156]:
df5 = pd.merge(df4, adj_close, on=['YF_Symbol', 'Date'], how='left')

#### Price data by day

In [157]:
def daily_data(df):
    df.dropna(inplace = True)
    price = df.stack(level=1).reset_index(level=0)
    price.reset_index(inplace=True)
    price.rename(columns= {"index": "Ticker"}, inplace=True)
    price = price[['Ticker', 'Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']]
    price.rename(columns = {"Ticker" : "YF_Symbol"}, inplace=True)
    price = pd.merge(price, mapping_df, on="YF_Symbol", how='left')
    price.sort_values(["YF_Symbol", "Date"], inplace = True)
    price.reset_index(inplace = True)
    price = price[['Market','YF_Symbol', 'Date', 'Adj Close', 'Close', 'High', 'Low', 'Open',
       'Volume', 'group']]
    price["200MA"] = price.groupby("YF_Symbol")["Adj Close"].rolling(window=200).mean().reset_index(0, drop=True)
    
    # Calculate RSI for each symbol
    price['RSI'] = price.groupby('YF_Symbol')['Adj Close'].transform(lambda x: rsi(x))

    return price

In [158]:
price_data = pd.merge(df4, adj_close, on=['YF_Symbol', 'Date'], how='right')

In [159]:
price_data.dropna(subset=['Close'], inplace = True)

#### df 5 now contains price and open interest by week

In [160]:
df5["200MA"] = df5.groupby("Market")["Close"].rolling(window=28).mean().reset_index(0, drop=True)
#its 28 because data is weekly and there are 28 weeks in 200 days

In [161]:
df5 = df5.sort_values(['Market', 'Date'], ascending = [True, True])

In [162]:
df5['RSI'] = df5.groupby('Market')['Close'].transform(lambda x: rsi(x))

In [163]:
#Put the data above into a dataframe without the COT data for markets we are not trading (i.e removing everything not in unique markets)

df6 = []  # Initialize an empty list to store filtered dataframes

for market_name in unique_markets:
    df6.append(df5[df5["Market"] == market_name])

# Concatenate the filtered dataframes into a single DataFrame
df6 = pd.concat(df6, ignore_index=True)


        

In [164]:
# Export the combined dataset (both weekly COT data and daily price data)
df6_combined.to_json('cot_data.json', orient='records')
print("Combined data (weekly COT + daily prices) exported to cot_data.json")
print(f"Total records exported: {len(df6_combined)}")

Combined data (weekly COT + daily prices) exported to cot_data.json
Total records exported: 50109


In [57]:
df6.iloc[-1]

Market                                     VIX FUTURES - CBOE FUTURES EXCHANGE
Date                                                       2025-07-22 00:00:00
Noncommercial Positions-Long (All)                                      102963
Noncommercial Positions-Short (All)                                     153558
Noncommercial Positions-Spreading (All)                                  79130
Commercial Positions-Long (All)                                         190444
Commercial Positions-Short (All)                                        141491
 Total Reportable Positions-Long (All)                                  372537
Total Reportable Positions-Short (All)                                  374179
Nonreportable Positions-Long (All)                                       25933
Nonreportable Positions-Short (All)                                      24291
OI_Index                                                             50.855221
OI                                                  

#### Creating a DashBoard

In [54]:
import plotly.offline as py
import plotly.graph_objs as go
import plotly.express as px
import dash
from dash import Dash, Input, Output, State, html, dcc, callback, dash_table
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
from jupyter_dash import JupyterDash
import dash_bootstrap_components as dbc
from datetime import date, datetime, timedelta

external_stylesheets = [dbc.themes.CYBORG]

app = Dash(__name__, title = "Interactive Dashboard", external_stylesheets=external_stylesheets)

columns_format = ['OI_Index', 'Retail_Index', 'Commercial_Index']
df6['Date'] = pd.to_datetime(df6['Date'])
week_dates = df6['Date'].dt.strftime('%Y-%m-%d').unique().tolist()
days_ago = datetime.now() - timedelta(days=200) #this is for loading the price graph
future_graph_space = datetime.now() + timedelta(days=15) #this is to limit future x axis of the price graph


In [55]:
app.layout = dbc.Container([
    # Title Row
    dbc.Row([
        dbc.Col(
            html.H1("Commodity Prices and Open Interest Dashboard",
                   className="text-center mb-4 mt-3",
                   style={'color': 'white'})
        )
    ]),

    # Price & Open Interest Analysis Section
    dbc.Row([
        dbc.Col([
            html.Div("Price & Open Interest Analysis",
                style={'textAlign': 'center', 'color': 'white', 'fontSize': 20, 'marginBottom': '20px'}),
        # Container for dropdown and label
            html.Div([
                html.Label('Select Market:', style={'color': 'white', 'fontSize': 16}),
                dcc.Dropdown(
                id='commodity-dropdown',
                options=[{'label': market, 'value': market} for market in df6['Market'].unique()],
                value=df6['Market'].unique()[0],
                style={'width': '300px', 'margin': '0 auto'}  # Fixed width and centered
            )
        ], style={'textAlign': 'center', 'marginBottom': '30px'}),  # Added spacing
        # Container for the graph
            html.Div([
                dcc.Graph(id='combined-graph')
            ], style={'width': '90%', 'margin': '0 auto'})  # Centered with some margin
        ])
    ]),

    # Buy and Sell Indications Section
    dbc.Row([
        dbc.Col([
            html.Div("Buy and Sell Indications", 
                    style={'textAlign': 'center', 'color': 'white', 'fontSize': 20, 'marginBottom': '20px'}),
            dash_table.DataTable(
                id='table',
                data=summary_table_filtered.to_dict('records'),
                columns=[{'name': col, 'id': col, 'type': 'numeric', 'format': {'specifier': '.0f'}}
                        if summary_table_filtered[col].dtype in ['float64', 'float32']
                        else {'name': col, 'id': col}
                        for col in summary_table_filtered.columns],
                style_table={'maxWidth': '800px', 'margin': 'auto', 'overflowx': 'auto'},  # Increased maxWidth
                style_cell={'textAlign': 'center', 'color': 'black', 'fontSize': 12, 'padding': '10px', 
                           'minWidth': '70px', 'width': '100px', 'maxWidth': '180px'},
                style_header={'backgroundColor': '#2c3e50', 'color': 'white', 'fontWeight': 'bold', 
                            'textAlign': 'center', 'fontSize': 14},
                style_data_conditional=[
                    {
                        'if': {'filter_query': '{OI_Index} >= 80', 'column_id': 'OI_Index'},
                        'backgroundColor': '#4D9F6B',
                        'color': 'white'
                    },
                    {
                        'if': {'filter_query': '{OI_Index} <= 20', 'column_id': 'OI_Index'},
                        'backgroundColor': '#C46A33',
                        'color': 'white'
                    },
                    {
                        'if': {'filter_query': '{Retail_Index} >= 80', 'column_id': 'Retail_Index'},
                        'backgroundColor': '#4D9F6B',
                        'color': 'white'
                    },
                    {
                        'if': {'filter_query': '{Retail_Index} <= 20', 'column_id': 'Retail_Index'},
                        'backgroundColor': '#C46A33',
                        'color': 'white'
                    },
                    {
                        'if': {'filter_query': '{Commercial_Index} >= 80', 'column_id': 'Commercial_Index'},
                        'backgroundColor': '#4D9F6B',
                        'color': 'white'
                    },
                    {
                        'if': {'filter_query': '{Commercial_Index} <= 20', 'column_id': 'Commercial_Index'},
                        'backgroundColor': '#C46A33',
                        'color': 'white'
                    }
                ]
            )
        ])
    ]),

    # Bubble Chart Section
    dbc.Row([
        dbc.Col([
            html.Div("Retail vs Commercial Positioning",
                    style={'textAlign': 'center', 'color': 'white', 'fontSize': 20, 
                           'marginTop': '20px', 'marginBottom': '20px'}),
            html.Label("Select A Tuesday Date", style={'color': 'white'}),
            dcc.DatePickerSingle(
                id='my-date-picker-single',
                min_date_allowed=min(week_dates),
                max_date_allowed=max(week_dates),
                initial_visible_month=max(week_dates),
                date=max(week_dates),
                style={'fontSize': 14}
            ),
            dcc.Graph(id='bubble-graph')
        ])
    ]),

    # Open Interest Graph Section
    dbc.Row([
        dbc.Col([
            html.Div("Open Interest Index",
                    style={'textAlign': 'center', 'color': 'white', 'fontSize': 20, 
                           'marginTop': '20px', 'marginBottom': '20px'}),
            dcc.Graph(id='open-interest-graph')
        ])
    ])
])

# Callback for bubble graph
@app.callback(
    Output('bubble-graph', 'figure'),
    Input('my-date-picker-single', 'date')
)

def update_bubble(week_selected):
    
    # Get all data up to the selected date
    if isinstance(week_selected, str):
        week_selected = pd.to_datetime(week_selected).date()
    else:
        week_selected = pd.to_datetime(week_selected).date()
    
    date_selected = df6[df6["Date"].dt.date <= week_selected]
    
    fig = px.scatter(date_selected, 
                    x='Retail_Index', 
                    y='Commercial_Index', 
                    hover_data=['Market', 'Date'],
                    color='group', 
                    size='OI', 
                    size_max=40,
                    animation_frame='Date',
                    animation_group='Market',
                    labels={'Retail_Index': 'Retail', 
                           'Commercial_Index': 'Commercial Index'}
                   )

    # Set x-axis and y-axis ranges for each subplot
    fig.update_xaxes(range=[0, 125])
    fig.update_yaxes(range=[0, 125])
    
    # Update layout for better animation display
    fig.update_layout(
        height=800,
        width=1200,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="middle",
            y=1.10,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="lightgrey",
            borderwidth=1
        ),
        margin=dict(r=150),  
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[dict(
                label="Play",
                method="animate",
                args=[None, {"frame": {"duration": 500, "redraw": True},
                           "fromcurrent": True}]
            ),
            dict(
                label="Pause",
                method="animate",
                args=[[None], {"frame": {"duration": 0, "redraw": True},
                             "mode": "immediate",
                             "transition": {"duration": 0}}]
            )]
        )]
    )
    
    return fig

# Callback to update open interest graph
@app.callback(
    Output('open-interest-graph', 'figure'),
    [Input('commodity-dropdown', 'value'),
     Input('combined-graph', 'relayoutData')]
)
def update_open_interest_graph(selected_commodities, relayout_data):
    if not selected_commodities:
        return {}
        
    data = df6[df6['Market'] == selected_commodities]
    x = data['Date']
    y = data['OI_Index']
    
    fig = go.Figure()
    
    # Add the main trace
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines',
        name='Open Interest Index',
        line=dict(color='#1f77b4', width=2)
    ))
    
    # Add horizontal lines
    high_value = 80
    low_value = 20
    
    fig.update_layout(
        shapes=[
            dict(
                type='line',
                x0=min(x),
                x1=max(x),
                y0=high_value,
                y1=high_value,
                line=dict(color='orange', dash='dash')
            ),
            dict(
                type='line',
                x0=min(x),
                x1=max(x),
                y0=low_value,
                y1=low_value,
                line=dict(color='orange', dash='dash')
            )
        ],
        title=f'Open Interest Index: {selected_commodities}',
        xaxis_title='Date',
        yaxis_title='Index Value',
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        height=400,
        margin=dict(t=50, b=50, l=50, r=50),
        xaxis=dict(
            rangeslider_visible=False,
            rangeselector=dict(
                buttons=[
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(count=6, label="6m", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(count=1, label="1y", step="year", stepmode="backward"),
                    dict(step="all")
                ],
                y=0.95,
                yanchor="bottom",
                x=0.5,
                xanchor="center",
                bgcolor="white",
                bordercolor="lightgray",
                borderwidth=1,
                font=dict(size=10)
            )
        )
    )
    
    return fig
# New callback for the combined price and open interest graph
@app.callback(
    Output('combined-graph', 'figure'),
    Input('commodity-dropdown', 'value')
)
def update_combined_graph(selected_commodities):
    
    data = df6[df6['Market'] == selected_commodities]
    
    # Create figure with three rows (panes)
    fig = make_subplots(rows=3, cols=1, 
                        shared_xaxes=True, 
                        vertical_spacing=0.1,
                        row_heights=[0.5, 0.25, 0.25],
                        subplot_titles=(f"Price: {selected_commodities}", 
                                      "Commercial & Retail Open Interest Indices",
                                      "RSI Technical Indicator"))

    # Add price trace on top pane
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data['Close'],
            name="Price",
            line=dict(color="#1f77b4", width=2)
        ),
        row=1, col=1
    )
    
    # Add 200-day moving average if it exists
    if '200MA' in data.columns:
        fig.add_trace(
            go.Scatter(
                x=data['Date'],
                y=data['200MA'],
                name="200-day MA",
                line=dict(color="orange", width=1.5)
            ),
            row=1, col=1
        )

    # Add commercial open interest trace on middle pane
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data['Commercial_Index'],
            name="Commercial Index",
            line=dict(color="#2ca02c", width=2)
        ),
        row=2, col=1
    )

    # Add retail open interest trace on middle pane
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data['Retail_Index'],
            name="Retail Index",
            line=dict(color="red", width=2)
        ),
        row=2, col=1
    )

    # Add RSI trace on bottom pane
    if 'RSI' in data.columns:
        fig.add_trace(
            go.Scatter(
                x=data['Date'],
                y=data['RSI'],
                name="RSI",
                line=dict(color="purple", width=2)
            ),
            row=3, col=1
        )

    # Add horizontal lines at 20 and 80 for the indices on middle pane
    fig.add_shape(
        type="line", line=dict(color="orange", width=1, dash="dash"),
        y0=20, y1=20, x0=data['Date'].min(), x1=data['Date'].max(),
        row=2, col=1
    )
    fig.add_shape(
        type="line", line=dict(color="orange", width=1, dash="dash"),
        y0=80, y1=80, x0=data['Date'].min(), x1=data['Date'].max(),
        row=2, col=1
    )

    # Add horizontal lines at 30 and 70 for RSI on bottom pane
    fig.add_shape(
        type="line", line=dict(color="gray", width=1, dash="dash"),
        y0=30, y1=30, x0=data['Date'].min(), x1=data['Date'].max(),
        row=3, col=1
    )
    fig.add_shape(
        type="line", line=dict(color="gray", width=1, dash="dash"),
        y0=70, y1=70, x0=data['Date'].min(), x1=data['Date'].max(),
        row=3, col=1
    )

    # Set y-axes titles and ranges
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Index Value", range=[0, 100], row=2, col=1)
    fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)

    # Update x-axis with range selector on bottom pane
    fig.update_xaxes(
        rangeslider_visible=False,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ]),
            y=0.95,  # Position at bottom
            yanchor="bottom",  # Anchor to bottom
            x=0.5,  # Center horizontally
            xanchor="center",  # Anchor to center
            bgcolor="white",  # White background
            bordercolor="lightgray",  # Light gray border
            borderwidth=1,  # Border width
            font=dict(size=10)  # Smaller font size
        ),
        row=3, col=1
    )

    # Update x-axis title on bottom pane
    fig.update_xaxes(title_text="Date", row=3, col=1)
    
    # Set overall layout
    fig.update_layout(
        height=1000,
        width=1000,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white"
    )

    return fig

In [56]:
if __name__ == '__main__':
    app.run_server(jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


/Applications/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result

/Applications/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result

/Applications/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [ ]:
app._terminate_server_for_port("localhost", 8050)

#Tip for major issues terminating port if above fails: 
# Terminal
# sudo lsof -i :8050
# kill -9 xxxxx replace xxxxx with the PID number

